# APCS 初級模式｜ZeroJudge 七題實作診斷

## 開始之前｜如何使用

1. 在 **Jupyter Notebook / JupyterLab** 開啟，使用 **Python 3** kernel；程式本體仍以 **C11** 撰寫。
2. 先執行下方「測試器初始化」程式格；需要本機有 `cc`／`clang`／`gcc`。
3. 每題先閱讀摘要，**再點 ZeroJudge 原題確認完整規格**，自行規劃解題方法。
4. 在每題的 `%%writefile` 格寫 C 程式並執行（這一步會產生 `.c` 檔）；然後執行 `run_public("題號")` 查看本機公開測試結果。
5. 修改完 C 程式後，務必**再次執行 `%%writefile` 格**，才會更新檔案。正式提交 ZeroJudge 時，只貼 `.c` 檔裡的 C 程式，**不要貼 `%%writefile`**。
6. 不需要 Python 第三方套件；測試器只使用標準函式庫。`PASS` 只代表本機題庫通過，**不代表 ZeroJudge 已 AC**。

> 請注意：Notebook 的 Python 測試碼不是要你用 Python 解題。`%%writefile` 是 Jupyter 的寫檔指令，下面接著寫標準 C 程式。


## 初始化｜Python 測試器（使用本機 C 編譯器）


In [ ]:
# 【共用測試器】只需要 Python 標準函式庫 + 本機 C 編譯器
import subprocess, shutil, tempfile, pathlib, time, json, csv, random, math
from pathlib import Path

PROBLEM_IDS = ["a006", "c294", "a034", "a024", "a022", "a104", "b964"]
PUBLIC = {'a006': [('1 3 -10\n', 'Two different roots x1=2 , x2=-5\n'), ('1 0 0\n', 'Two same roots x=0\n'), ('1 1 1\n', 'No real root\n')], 'c294': [('3 4 5\n', '3 4 5\nRight\n'), ('101 100 99\n', '99 100 101\nAcute\n'), ('10 10 100\n', '10 10 100\nNo\n')], 'a034': [('3\n6\n', '11\n110\n'), ('0\n1\n', '0\n1\n'), ('15\n255\n', '1111\n11111111\n')], 'a024': [('12 15\n', '3\n'), ('1 100\n', '1\n'), ('144 60\n', '12\n')], 'a022': [('abba\n', 'yes\n'), ('abcd\n', 'no\n'), ('x\n', 'yes\n')], 'a104': [('6\n7 9 0 4 1 8\n8\n1 9 9 0 0 9 2 8\n', '0 1 4 7 8 9\n0 0 1 2 8 9 9 9\n'), ('1\n42\n', '42\n'), ('5\n5 4 3 2 1\n', '1 2 3 4 5\n')], 'b964': [('10\n0 11 22 33 55 66 77 99 88 44\n', '0 11 22 33 44 55 66 77 88 99\n55\n66\n'), ('1\n13\n', '13\n13\nworst case\n'), ('2\n73 65\n', '65 73\nbest case\n65\n')]}
COMPILER = shutil.which("cc") or shutil.which("clang") or shutil.which("gcc")
STUDENT_DIR = Path("apcs_junior_student_answers")
REFERENCE_DIR = Path("apcs_junior_teacher_reference")
STUDENT_DIR.mkdir(parents=True, exist_ok=True)
REFERENCE_DIR.mkdir(parents=True, exist_ok=True)
print("C 編譯器：", COMPILER if COMPILER else "未找到。請先安裝 clang / gcc。")
print("學生作答目錄：", STUDENT_DIR.resolve())

def _clean(s):
    # 每行忽略尾端空格與檔尾空白，但保留行數與行內文字，避免遮住重要格式錯誤
    return "\n".join(line.rstrip() for line in s.rstrip().splitlines())

def _evaluate(problem_id, cases, src_dir=STUDENT_DIR, show_first_error=True):
    if problem_id not in PROBLEM_IDS:
        raise ValueError("不認得題號：" + str(problem_id))
    path = Path(src_dir) / (problem_id + ".c")
    if not path.exists():
        print("找不到程式檔：", path, "\n請先執行題目的 %%writefile 儲存格。")
        return {"id": problem_id, "passed": 0, "total": len(cases), "status": "MISSING"}
    if not COMPILER:
        print("找不到 C 編譯器，請先安裝 cc / clang / gcc。")
        return {"id": problem_id, "passed": 0, "total": len(cases), "status": "NO_COMPILER"}
    with tempfile.TemporaryDirectory(prefix="apcs7_") as td:
        exe = Path(td) / "answer"
        compilation = subprocess.run([COMPILER, "-std=c11", "-O2", "-Wall", "-Wextra", str(path), "-o", str(exe), "-lm"], capture_output=True, text=True)
        if compilation.returncode != 0:
            print(f"[{problem_id}] CE｜編譯錯誤\n", compilation.stderr[:2500])
            return {"id": problem_id, "passed": 0, "total": len(cases), "status": "CE"}
        passed=0
        for i, (stdin, expected) in enumerate(cases, 1):
            try:
                result = subprocess.run([str(exe)], input=stdin, capture_output=True, text=True, timeout=2)
            except subprocess.TimeoutExpired:
                if show_first_error: print(f"[{problem_id}] TLE｜第 {i} 組超過 2 秒。")
                break
            if result.returncode != 0:
                if show_first_error: print(f"[{problem_id}] RE｜第 {i} 組執行失敗，代碼 {result.returncode}\n{result.stderr[:700]}")
                break
            if _clean(result.stdout)!=_clean(expected):
                if show_first_error:
                    print(f"[{problem_id}] WA｜第 {i} / {len(cases)} 組不符")
                    print("輸入：", repr(stdin[:400]))
                    print("期望：", repr(expected[:500]))
                    print("實際：", repr(result.stdout[:500]))
                break
            passed += 1
        status = "PASS" if passed==len(cases) else "WA/TLE/RE"
        print(f"[{problem_id}] {status}｜通過 {passed}/{len(cases)} 組本機測試")
        return {"id":problem_id,"passed":passed,"total":len(cases),"status":status}

def run_public(problem_id):
    """學生版公開測資；不代表 ZeroJudge 正式 AC。"""
    return _evaluate(problem_id, PUBLIC[problem_id], STUDENT_DIR)


## 題目總覽

| 題號 | 題名 | 診斷重點 | 建議時間 |
|---|---|---|---|
| a006 | [一元二次方程式](https://zerojudge.tw/ShowProblem?problemid=a006) | 判別式、三種情況與精確輸出 | 10–15 分 |
| c294 | [三角形辨別](https://zerojudge.tw/ShowProblem?problemid=c294) | 排序、三角形不等式、多層條件 | 15–20 分 |
| a034 | [二進位制轉換](https://zerojudge.tw/ShowProblem?problemid=a034) | while、除以 2、反向輸出、EOF | 15–20 分 |
| a024 | [最大公因數 (GCD)](https://zerojudge.tw/ShowProblem?problemid=a024) | 模運算、輾轉相除法、終止條件 | 15–20 分 |
| a022 | [迴文](https://zerojudge.tw/ShowProblem?problemid=a022) | 字串、索引、雙向比較 | 15–20 分 |
| a104 | [排序](https://zerojudge.tw/ShowProblem?problemid=a104) | 陣列、排序、EOF、多筆測資 | 20–25 分 |
| b964 | [成績指標](https://zerojudge.tw/ShowProblem?problemid=b964) | APCS 實作整合：排序、搜尋、邊界 | 25–30 分 |


---
## 第 1 題｜a006：一元二次方程式

**目標能力：** 判別式、三種情況與精確輸出　　**建議時間：** 10–15 分

**ZeroJudge 原題：** [a006 — 一元二次方程式](https://zerojudge.tw/ShowProblem?problemid=a006)

**題意簡述：** 給定二次方程式的三個整數係數 a、b、c，判定實根數量，並依指定格式輸出根（如有兩個根，大的先輸出）。

**輸入**：一行三個整數 `a b c`。**輸出**：依題目規定使用 `Two different roots x1=... , x2=...`、`Two same roots x=...` 或 `No real root`。

**練習範例輸入**
```text
1 3 -10
```
**練習範例輸出**
```text
Two different roots x1=2 , x2=-5
```

**作答前請先想：** 讀取幾筆資料？要用哪個迴圈或條件？哪些輸入最容易讓程式出錯？


<details>
<summary>需要時才展開：提示 1</summary>

先計算 `D = b*b - 4*a*c`。請列出 D 大於、等於、小於 0 三種分支。

</details>

<details>
<summary>需要時才展開：提示 2</summary>

有兩個根時要「由大到小」輸出；留意 `a` 可能不是正數，且格式字串必須完全符合題目。

</details>


### 學生 C11 作答區｜先執行這格，再執行下一格測試


In [ ]:
%%writefile apcs_junior_student_answers/a006.c
#include <stdio.h>

int main(void) {
    /* TODO: 讀入、運算、輸出。請刪除本註解並完成程式。 */
    return 0;
}


In [ ]:
run_public("a006")


**自我檢查**：☐ 有獨立完成　☐ 有測過不同輸入　☐ 知道 WA／CE 的原因　☐ 已到 ZeroJudge 正式提交


---
## 第 2 題｜c294：三角形辨別

**目標能力：** 排序、三角形不等式、多層條件　　**建議時間：** 15–20 分

**ZeroJudge 原題：** [c294 — 三角形辨別](https://zerojudge.tw/ShowProblem?problemid=c294)

**題意簡述：** 讀入三個正整數邊長；先依小到大輸出，再判定是否能構成三角形。若能，區分銳角、直角、鈍角。

**輸入**：一行 `a b c`。**輸出**：第一行三邊升序，第二行為 `No`、`Acute`、`Right` 或 `Obtuse`。

**練習範例輸入**
```text
3 4 5
```
**練習範例輸出**
```text
3 4 5
Right
```

**作答前請先想：** 讀取幾筆資料？要用哪個迴圈或條件？哪些輸入最容易讓程式出錯？


<details>
<summary>需要時才展開：提示 1</summary>

先把三個數排成 `x <= y <= z`；判斷是否能成三角形必須在角度判斷之前。

</details>

<details>
<summary>需要時才展開：提示 2</summary>

比較 `x*x + y*y` 與 `z*z`，不要用 `sqrt`；注意邊長接近 30000 時的整數安全性。

</details>


### 學生 C11 作答區｜先執行這格，再執行下一格測試


In [ ]:
%%writefile apcs_junior_student_answers/c294.c
#include <stdio.h>

int main(void) {
    /* TODO: 讀入、運算、輸出。請刪除本註解並完成程式。 */
    return 0;
}


In [ ]:
run_public("c294")


**自我檢查**：☐ 有獨立完成　☐ 有測過不同輸入　☐ 知道 WA／CE 的原因　☐ 已到 ZeroJudge 正式提交


---
## 第 3 題｜a034：二進位制轉換

**目標能力：** while、除以 2、反向輸出、EOF　　**建議時間：** 15–20 分

**ZeroJudge 原題：** [a034 — 二進位制轉換](https://zerojudge.tw/ShowProblem?problemid=a034)

**題意簡述：** 將輸入的每個十進位非負整數轉為二進位字串；資料會一直輸入到檔案結尾。

**輸入**：多行十進位整數，直到 EOF。**輸出**：每個輸入對應一行二進位表示。

**練習範例輸入**
```text
3
6
```
**練習範例輸出**
```text
11
110
```

**作答前請先想：** 讀取幾筆資料？要用哪個迴圈或條件？哪些輸入最容易讓程式出錯？


<details>
<summary>需要時才展開：提示 1</summary>

用 `% 2` 取最低位、`/ 2` 刪掉最低位，但這會先得到低位到高位。

</details>

<details>
<summary>需要時才展開：提示 2</summary>

可先把每個位數存陣列，再由後往前輸出；讀取多組資料時可用 `while (scanf(...) == 1)`。

</details>


### 學生 C11 作答區｜先執行這格，再執行下一格測試


In [ ]:
%%writefile apcs_junior_student_answers/a034.c
#include <stdio.h>

int main(void) {
    /* TODO: 讀入、運算、輸出。請刪除本註解並完成程式。 */
    return 0;
}


In [ ]:
run_public("a034")


**自我檢查**：☐ 有獨立完成　☐ 有測過不同輸入　☐ 知道 WA／CE 的原因　☐ 已到 ZeroJudge 正式提交


---
## 第 4 題｜a024：最大公因數 (GCD)

**目標能力：** 模運算、輾轉相除法、終止條件　　**建議時間：** 15–20 分

**ZeroJudge 原題：** [a024 — 最大公因數 (GCD)](https://zerojudge.tw/ShowProblem?problemid=a024)

**題意簡述：** 輸入兩個正整數，求出最大公因數；不能只靠手動列因數，必須提出可重複的演算法。

**輸入**：兩個正整數。**輸出**：它們的最大公因數。

**練習範例輸入**
```text
12 15
```
**練習範例輸出**
```text
3
```

**作答前請先想：** 讀取幾筆資料？要用哪個迴圈或條件？哪些輸入最容易讓程式出錯？


<details>
<summary>需要時才展開：提示 1</summary>

試算 `gcd(48,18)`，觀察 `48 % 18`、`18 % 12`、`12 % 6`。

</details>

<details>
<summary>需要時才展開：提示 2</summary>

每次把 `(a,b)` 更新成 `(b,a%b)`，直到某個數變為 0。

</details>


### 學生 C11 作答區｜先執行這格，再執行下一格測試


In [ ]:
%%writefile apcs_junior_student_answers/a024.c
#include <stdio.h>

int main(void) {
    /* TODO: 讀入、運算、輸出。請刪除本註解並完成程式。 */
    return 0;
}


In [ ]:
run_public("a024")


**自我檢查**：☐ 有獨立完成　☐ 有測過不同輸入　☐ 知道 WA／CE 的原因　☐ 已到 ZeroJudge 正式提交


---
## 第 5 題｜a022：迴文

**目標能力：** 字串、索引、雙向比較　　**建議時間：** 15–20 分

**ZeroJudge 原題：** [a022 — 迴文](https://zerojudge.tw/ShowProblem?problemid=a022)

**題意簡述：** 判斷一個不含空白的字串，從左往右和從右往左讀是否完全相同（大小寫須按字元原樣比較）。

**輸入**：一個長度小於 1000 的字串。**輸出**：`yes` 或 `no`（小寫）。

**練習範例輸入**
```text
abba
```
**練習範例輸出**
```text
yes
```

**作答前請先想：** 讀取幾筆資料？要用哪個迴圈或條件？哪些輸入最容易讓程式出錯？


<details>
<summary>需要時才展開：提示 1</summary>

一邊從左端、另一邊從右端開始比較；只要有一組不同就不是迴文。

</details>

<details>
<summary>需要時才展開：提示 2</summary>

要知道字串長度可以用 `strlen`；右端索引應從 `len-1` 開始。

</details>


### 學生 C11 作答區｜先執行這格，再執行下一格測試


In [ ]:
%%writefile apcs_junior_student_answers/a022.c
#include <stdio.h>

int main(void) {
    /* TODO: 讀入、運算、輸出。請刪除本註解並完成程式。 */
    return 0;
}


In [ ]:
run_public("a022")


**自我檢查**：☐ 有獨立完成　☐ 有測過不同輸入　☐ 知道 WA／CE 的原因　☐ 已到 ZeroJudge 正式提交


---
## 第 6 題｜a104：排序

**目標能力：** 陣列、排序、EOF、多筆測資　　**建議時間：** 20–25 分

**ZeroJudge 原題：** [a104 — 排序](https://zerojudge.tw/ShowProblem?problemid=a104)

**題意簡述：** 一組資料先給元素個數 n，再給 n 個整數；對每一組資料輸出由小到大排序的結果。可能有多組資料直到 EOF。

**輸入**：多組資料，每組先讀 `n` 再讀 `n` 個整數。**輸出**：每組一行升序數列，數字用空格隔開。

**練習範例輸入**
```text
6
7 9 0 4 1 8
```
**練習範例輸出**
```text
0 1 4 7 8 9
```

**作答前請先想：** 讀取幾筆資料？要用哪個迴圈或條件？哪些輸入最容易讓程式出錯？


<details>
<summary>需要時才展開：提示 1</summary>

可以先使用插入排序或氣泡排序，想清楚比較的索引範圍。

</details>

<details>
<summary>需要時才展開：提示 2</summary>

最後一個數字後不要多印空格；每一組資料都要重新讀取 n 個元素。

</details>


### 學生 C11 作答區｜先執行這格，再執行下一格測試


In [ ]:
%%writefile apcs_junior_student_answers/a104.c
#include <stdio.h>

int main(void) {
    /* TODO: 讀入、運算、輸出。請刪除本註解並完成程式。 */
    return 0;
}


In [ ]:
run_public("a104")


**自我檢查**：☐ 有獨立完成　☐ 有測過不同輸入　☐ 知道 WA／CE 的原因　☐ 已到 ZeroJudge 正式提交


---
## 第 7 題｜b964：成績指標

**目標能力：** APCS 實作整合：排序、搜尋、邊界　　**建議時間：** 25–30 分

**ZeroJudge 原題：** [b964 — 成績指標](https://zerojudge.tw/ShowProblem?problemid=b964)

**題意簡述：** 將班級成績升序輸出，再找最高的不及格分數（低於 60）與最低的及格分數（至少 60）。若某群不存在，需輸出特別字串。

**輸入**：`n` 與 n 個 0～100 的分數（n 為 1～20）。**輸出**：三行：排序結果、最高不及格分數或 `best case`、最低及格分數或 `worst case`。

**練習範例輸入**
```text
10
0 11 22 33 55 66 77 99 88 44
```
**練習範例輸出**
```text
0 11 22 33 44 55 66 77 88 99
55
66
```

**作答前請先想：** 讀取幾筆資料？要用哪個迴圈或條件？哪些輸入最容易讓程式出錯？


<details>
<summary>需要時才展開：提示 1</summary>

先排序；之後確認「低於 60 的最後一個」與「不低於 60 的第一個」在哪裡。

</details>

<details>
<summary>需要時才展開：提示 2</summary>

三類都要測：有人及格且有人不及格、全部及格、全部不及格。不要讀取不存在的陣列位置。

</details>


### 學生 C11 作答區｜先執行這格，再執行下一格測試


In [ ]:
%%writefile apcs_junior_student_answers/b964.c
#include <stdio.h>

int main(void) {
    /* TODO: 讀入、運算、輸出。請刪除本註解並完成程式。 */
    return 0;
}


In [ ]:
run_public("b964")


**自我檢查**：☐ 有獨立完成　☐ 有測過不同輸入　☐ 知道 WA／CE 的原因　☐ 已到 ZeroJudge 正式提交


---
## 學生完成紀錄

| 題號 | 已做 / 未做 | 本機測試 | ZeroJudge 狀態 | 我最容易錯的地方 |
|---|---|---|---|---|
| a006 |  |  |  |  |
| c294 |  |  |  |  |
| a034 |  |  |  |  |
| a024 |  |  |  |  |
| a022 |  |  |  |  |
| a104 |  |  |  |  |
| b964 |  |  |  |  |


## 交作業前

1. 確定每一題都已重新執行 `%%writefile`，程式碼已寫入檔案。
2. 將完成的 `.c` 檔與 ZeroJudge 提交結果交給老師。
3. 寫下：**哪一題卡最久？為什麼？** 以及 **你自己設計的至少一組邊界測資**。
